In [2]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your dataset
with open("data.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# We'll start with subject classification
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["subject"], test_size=0.2, random_state=42
)


In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

from sklearn.preprocessing import LabelEncoder
import torch

label_encoder = LabelEncoder()
train_labels_enc = label_encoder.fit_transform(train_labels)
test_labels_enc = label_encoder.transform(test_labels)

class NotesDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NotesDataset(train_encodings, train_labels_enc)
test_dataset = NotesDataset(test_encodings, test_labels_enc)


C:\Users\Parushi\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [4]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/longformer-base-4096",
    num_labels=len(label_encoder.classes_)
)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()


Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Parushi\AppData\Roaming\Python\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Epoch,Training Loss,Validation Loss
1,No log,2.417660
2,No log,1.786143
3,No log,1.370326
4,No log,1.243293
5,No log,1.140895


TrainOutput(global_step=115, training_loss=1.4687839673913043, metrics={'train_runtime': 2774.797, 'train_samples_per_second': 0.16, 'train_steps_per_second': 0.041, 'total_flos': 61662578860080.0, 'train_loss': 1.4687839673913043, 'epoch': 5.0})

In [5]:
preds_output = trainer.predict(test_dataset)
pred_labels = preds_output.predictions.argmax(axis=1)

import numpy as np
from sklearn.metrics import classification_report

print(classification_report(
    test_labels_enc,
    pred_labels,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))



                      precision    recall  f1-score   support

               ADBMS       0.00      0.00      0.00         1
                  AI       0.67      1.00      0.80         2
             Biology       1.00      1.00      1.00         3
           Chemistry       0.00      0.00      0.00         1
     Cloud Computing       0.25      1.00      0.40         1
    Computer Science       0.00      0.00      0.00         1
           Economics       0.00      0.00      0.00         0
             History       0.00      0.00      0.00         1
Information Security       0.00      0.00      0.00         4
          Literature       0.00      0.00      0.00         1
                  ML       0.00      0.00      0.00         1
         Mathematics       1.00      1.00      1.00         2
                 NLP       0.00      0.00      0.00         1
             Physics       0.80      1.00      0.89         4
Predictive Modelling       0.00      0.00      0.00         0

      

In [11]:
def predict_subject(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([pred])[0]

print(predict_subject("""A major and fundamental challenge in distributed systems is 
achieving consensus. This is the process of getting all nodes in a 
network to come to a provable agreement on a single value or state, 
and to do so reliably, even in the face of unpredictable failures or 
network delays. This problem is non-trivial because messages 
between nodes can be lost, reordered, or delayed, and the nodes 
themselves can crash or become disconnected from the network (a 
'network partition'). 
This is not an abstract problem; it is a fundamental requirement for 
nearly all fault-tolerant systems. For example, in cloud databases 
(ADBMS) that replicate data for high availability, all replicas must 
agree on the order of transactions. If one replica processes a 'write' 
before a 'delete' and another does the opposite, the database becomes 
inconsistent. Consensus is used to elect a single 'leader' or 'primary' 
node that dictates the one true order of operations. Similarly, in 
blockchain technology, the entire network of decentralized nodes 
must agree on the exact set of transactions to be included in the next 
block, and the order of those blocks in the chain, to prevent "double
spending" and ensure the immutability of the ledger. 
The Paxos and Raft algorithms are two of the most well-known 
protocols designed to solve this exact problem. Paxos, developed first, 
is foundational but notoriously difficult to understand and implement 
correctly. Raft was developed later with the explicit goal of being 
more understandable than Paxos, breaking the problem down into 
distinct parts: leader election, log replication, and safety. Both 
protocols provide a fault-tolerant mechanism to ensure that as long as 
a majority of nodes (a 'quorum') are able to communicate, the system 
as a whole can continue to make progress, elect a single leader, and 
reach a correct, unified decision. This use of a quorum is what 
prevents the catastrophic 'split-brain' scenario, where a network 
partition might otherwise cause two separate parts of the network to 
both believe they are in charge, leading to two different and 
conflicting versions of the truth."""))


ADBMS


In [7]:
model.save_pretrained("subject_model")
tokenizer.save_pretrained("subject_model")


('subject_model\\tokenizer_config.json',
 'subject_model\\special_tokens_map.json',
 'subject_model\\vocab.json',
 'subject_model\\merges.txt',
 'subject_model\\added_tokens.json',
 'subject_model\\tokenizer.json')

In [8]:
import joblib
joblib.dump(label_encoder, "label_encoder.pkl")
model.save_pretrained("subject_model")
tokenizer.save_pretrained("subject_model")


('subject_model\\tokenizer_config.json',
 'subject_model\\special_tokens_map.json',
 'subject_model\\vocab.json',
 'subject_model\\merges.txt',
 'subject_model\\added_tokens.json',
 'subject_model\\tokenizer.json')